# The model you can actually read

MichAl Academy, lesson 2.4.

Run each cell with **Shift+Enter**.

Lesson 0.2 said you cannot read a model to know what it will do. That is true of
almost every model in this course. This notebook builds the exceptions, and they
are still in production everywhere.


## 1. Predicting a number

The diabetes dataset: 442 patients, ten measurements each, and a score for how
far the disease progressed over the following year. Ask it to be loaded
unscaled, so the numbers keep their units and the model stays readable.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score

diabetes = load_diabetes(scaled=False)
frame = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)

print(frame[["age", "bmi", "bp"]].describe().loc[["min", "mean", "max"]].round(1).to_string())
print()
print(f"score runs {diabetes.target.min():.0f} to {diabetes.target.max():.0f}, mean {diabetes.target.mean():.1f}")


In [ ]:
KF = KFold(n_splits=5, shuffle=True, random_state=0)

bmi = frame[["bmi"]].to_numpy()
line = LinearRegression().fit(bmi, diabetes.target)

print(f"score = {line.coef_[0]:.2f} * bmi + ({line.intercept_:.1f})")
print(f"r-squared, cross-validated: {cross_val_score(LinearRegression(), bmi, diabetes.target, cv=KF).mean():.3f}")


That is the entire model. Two numbers, and you can evaluate it with a
calculator: a body mass index of 28 predicts a score of about 168.

`r-squared` is the share of the variation the model accounts for, so 0.333 means
it explains a third of what is going on and misses two thirds. A real model, and
a modest one. Being able to read it does not make it right.


In [ ]:
for value in (20, 25, 30, 35):
    print(f"bmi {value}: {line.coef_[0]:.2f} * {value} + ({line.intercept_:.1f}) = {line.coef_[0] * value + line.intercept_:.1f}")


## 2. The coefficient trap

Now use all ten measurements instead of one.


In [ ]:
full = LinearRegression().fit(diabetes.data, diabetes.target)
print(f"r-squared with all ten: {cross_val_score(LinearRegression(), diabetes.data, diabetes.target, cv=KF).mean():.3f}")
print()

bmi_index = diabetes.feature_names.index("bmi")
print(f"bmi coefficient, on its own:            {line.coef_[0]:.2f}")
print(f"bmi coefficient, alongside nine others: {full.coef_[bmi_index]:.2f}")


The same measurement, the same data, and the number nearly halved.

Nothing is broken. A coefficient never means "the effect of this measurement".
It means "the effect of this measurement **once the others in this model are
held fixed**". Body mass index is related to blood pressure and to the blood
serum measurements, so when they are in the model too, they account for part of
what body mass index was previously credited with.

The consequence is worth stating plainly: **you cannot read a coefficient
without knowing what else is in the model.** Every claim of the form "the model
says X adds 10 points" is incomplete until you know what X was competing with.


## 3. Predicting a class

Same machinery, one extra step. Compute a weighted sum, then squash it into the
range 0 to 1 so it can be read as a probability.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

cancer = load_breast_cancer()
print(f"{cancer.data.shape[0]} samples, {cancer.data.shape[1]} measurements")
print(f"classes: {', '.join(cancer.target_names)}, and {cancer.target.mean():.1%} are benign")

SCV = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
print(f"accuracy: {cross_val_score(model, cancer.data, cancer.target, cv=SCV).mean():.3f}")


The squash is one line of arithmetic. Write it out rather than trusting it.


In [ ]:
def squash(z):
    return 1 / (1 + np.exp(-z))


for z in (-4, -1, 0, 1, 4):
    print(f"weighted sum {z:>3}  ->  probability {squash(z):.3f}")


Zero maps to exactly one half, which is why 0.5 is the natural cut. Large
positive sums go to nearly 1, large negative ones to nearly 0, and nothing ever
leaves the range. That is the whole difference between the two models.


## 4. Reading the weights

Fit it on the full data and look at what it decided. The features were scaled
first, so every coefficient is "per one standard deviation of that
measurement", which makes them comparable with each other.


In [ ]:
fitted = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(cancer.data, cancer.target)
weights = pd.Series(fitted[-1].coef_.ravel(), index=cancer.feature_names)

top = weights.reindex(weights.abs().sort_values(ascending=False).index).head(6)
print(f"{'measurement':>26} {'weight':>8} {'odds of benign':>16}")
for name, w in top.items():
    print(f"{name:>26} {w:>+8.3f} {'x' + format(np.exp(w), '.2f'):>16}")


Every one of the strongest weights is negative, and that is the model telling
you something true about the problem: larger and rougher measurements push
towards malignant, so they push away from benign.

The odds column is the readable form. A coefficient of -1.32 means that one
standard deviation more of that measurement multiplies the odds of benign by
about 0.27, so it cuts them to roughly a quarter.

This is what "interpretable" buys. Not a guarantee of correctness, but a
statement a specialist can look at and disagree with, which is the beginning of
catching the errors in lessons 2.3 and 2.10.


## 5. Naive Bayes, and the first spam filter that worked

One more model belongs here, because it is the historical answer to the first
machine learning problem most people ever met.

In 2002 Paul Graham argued you could filter spam "using nothing more than a
Bayesian combination of the spam probabilities of individual words". Build a
small one and watch it work. Six messages is enough.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

messages = [
    ("free money now, claim your free prize",   1),
    ("free offer, click now to claim",          1),
    ("win a free prize now",                    1),
    ("meeting at noon in the project room",     0),
    ("project update, please review the notes", 0),
    ("the noon meeting notes are attached",     0),
]

texts = [t for t, _ in messages]
spam = np.array([c for _, c in messages])

vectoriser = CountVectorizer()
counts = vectoriser.fit_transform(texts)
filter_ = MultinomialNB().fit(counts, spam)

print("vocabulary:", len(vectoriser.vocabulary_), "words")
for probe in ["free prize", "project meeting notes", "claim your project prize now"]:
    p = filter_.predict_proba(vectoriser.transform([probe]))[0, 1]
    print(f"  {probe:>30}  P(spam) = {p:.3f}")


Look at the third one. It contains `project`, which only ever appeared in
ordinary mail, and the filter still calls it spam with 96.8% confidence, because
`claim`, `prize` and `now` outvote it. Evidence combining is the whole idea.

Each word's contribution is visible:


In [ ]:
words = vectoriser.get_feature_names_out()
evidence = pd.Series(
    filter_.feature_log_prob_[1] - filter_.feature_log_prob_[0], index=words
).sort_values()

print("most ordinary")
print(evidence.head(4).round(3).to_string())
print()
print("most spammy")
print(evidence.tail(4).round(3).to_string())


### Why "naive"

The model assumes every word is independent of every other, given the class.
That is plainly false: "free" and "prize" turn up together far more often than
chance would suggest, and the model double-counts them as if they were separate
evidence.

It works anyway, which is the interesting part. Getting the probabilities
badly wrong often still gets the *ordering* right, and a filter only needs the
ordering. Graham reported missing fewer than 5 in 1,000 spams with no false
positives.

On tabular data it is usually beaten, though. Measure it:


In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.datasets import load_wine

for name, load in [("breast cancer", load_breast_cancer), ("wine", load_wine)]:
    X, y = load(return_X_y=True)
    logistic = cross_val_score(
        make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)), X, y, cv=SCV
    ).mean()
    bayes = cross_val_score(GaussianNB(), X, y, cv=SCV).mean()
    print(f"{name:>14}  logistic {logistic:.3f}   naive Bayes {bayes:.3f}")


Naive Bayes loses on both, and it is still worth knowing. It trains in one pass,
needs almost no memory, handles enormous vocabularies, and it is what is running
inside a great deal of software that has been quietly working since the early
2000s.


## What to take from this

| Model | Predicts | What you can read |
|---|---|---|
| Linear regression | A number | One weight per measurement, plus an intercept |
| Logistic regression | A probability | The same weights, as multipliers on the odds |
| Naive Bayes | A probability | One piece of evidence per feature |

Three things to carry forward.

**A coefficient depends on the company it keeps.** Body mass index was worth
10.23 alone and 5.6 alongside nine others. Never quote one without saying what
else was in the model.

**Interpretable does not mean correct.** The line explains a third of the
variation. It is readable and mostly wrong, and those are separate properties.

**Start here.** A logistic regression takes a minute to fit and tells you what
the problem looks like. If something more complicated cannot beat it, the
complication is not earning its place.
